In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
dataset_path = os.path.join(BASE_DIR, "hand_gesture_dataset_processed")

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir = os.path.join(dataset_path, "val")
test_dir = os.path.join(dataset_path, "test")

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=True
)

val_data = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=False
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(50, 50),
    batch_size=32,
    color_mode='grayscale',
    label_mode='categorical',
    shuffle=False
)

Found 529 files belonging to 8 classes.
Found 152 files belonging to 8 classes.
Found 76 files belonging to 8 classes.


In [ ]:
class_names = train_data.class_names
print(class_names)
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")

['0', '1', '2', '3', '4', 'Background', 'Left', 'Right']
Number of classes: 8


In [ ]:
with open(os.path.join(BASE_DIR, "class_indices.json"), "w") as f:
    json.dump(train_data.class_names, f)

In [ ]:
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(50, 50, 1)),  # Normalize pixel values to [0,1]
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.Flatten(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

# Display the architecture
model.summary()

c:\Users\louis\Documents\Telerobotics\computer_vision_speed_control\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 50, 50, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 50, 50, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 25, 25, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,179,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,204,104 (4.59 MB)

 Trainable params: 1,204,104 (4.59 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
checkpoint_path = os.path.join(BASE_DIR, "hand_gesture_model_best.keras")
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
earlystop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
callbacks = [checkpoint, earlystop]

In [ ]:
print("\nTraining the model...")
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)


Training the model...
Epoch 1/30
14/17 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2326 - loss: 2.0640
Epoch 1: val_loss improved from None to 1.29712, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 1: finished saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.4612 - loss: 1.8978 - val_accuracy: 0.7171 - val_loss: 1.2971
Epoch 2/30
15/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7339 - loss: 1.0704
Epoch 2: val_loss improved from 1.29712 to 0.36862, saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras

Epoch 2: finished saving model to c:\Users\louis\Documents\Telerobotics\hand_gesture_model_best.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7675 - loss: 0.8492 - val_accuracy: 0.9211 - val_loss: 0.3686
Epoch 3/30
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8796 - loss: 0.3387
Epoch 3: val_lo

In [ ]:
print("\nEvaluating on test data...")
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Evaluating on test data...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0013 
Test Loss: 0.0013
Test Accuracy: 1.0000


In [ ]:
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_data], axis=0)
preds = model.predict(test_data, verbose=1)
y_pred = np.argmax(preds, axis=1)

# Print and save confusion matrix and classification report
cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred, target_names=class_names)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", cr)
with open(os.path.join(BASE_DIR, "classification_report.txt"), "w") as f:
    f.write("Confusion Matrix:\n")
    f.write(str(cm))
    f.write("\n\nClassification Report:\n")
    f.write(cr)

# Plot confusion matrix (raw counts) and normalized version
tick_marks = np.arange(len(class_names))

# Raw confusion matrix heatmap
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm.max() / 2.0 if cm.size else 0
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, f"{cm[i, j]:d}", ha='center',
            color='white' if cm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusion_matrix.png'))
plt.close()

# Normalized confusion matrix (rows sum to 1)
cm_norm = cm.astype('float')
row_sums = cm_norm.sum(axis=1)[:, np.newaxis]
with np.errstate(divide='ignore', invalid='ignore'):
    cm_norm = np.divide(cm_norm, row_sums)
cm_norm = np.nan_to_num(cm_norm)

plt.figure(figsize=(8, 6))
plt.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Normalized Confusion Matrix')
plt.colorbar()
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
thresh = cm_norm.max() / 2.0 if cm_norm.size else 0
for i, j in np.ndindex(cm_norm.shape):
    plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha='center',
            color='white' if cm_norm[i, j] > thresh else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label (normalized)')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusion_matrix_normalized.png'))
plt.close()

# Plot training curves
plt.figure()
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.legend()
plt.title('Loss')
plt.savefig(os.path.join(BASE_DIR, 'loss_curve.png'))
plt.close()

plt.figure()
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.legend()
plt.title('Accuracy')
plt.savefig(os.path.join(BASE_DIR, 'accuracy_curve.png'))
plt.close()

# Save the trained model
model_save_path = os.path.join(BASE_DIR, "hand_gesture_model.keras")
model.save(model_save_path)
print(f"\nModel saved to {model_save_path}")

# Convert the model.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model.
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

Confusion Matrix:
 [[10  0  0  0  0  0  0  0]
 [ 0  9  0  0  0  0  0  0]
 [ 0  0  9  0  0  0  0  0]
 [ 0  0  0  9  0  0  0  0]
 [ 0  0  0  0 10  0  0  0]
 [ 0  0  0  0  0 10  0  0]
 [ 0  0  0  0  0  0  9  0]
 [ 0  0  0  0  0  0  0 10]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00         9
           3       1.00      1.00      1.00         9
           4       1.00      1.00      1.00        10
  Background       1.00      1.00      1.00        10
        Left       1.00      1.00      1.00         9
       Right       1.00      1.00      1.00        10

    accuracy                           1.00        76
   macro avg       1.00      1.00      1.00        76
weighted avg       1.00      1.00      1.00        76


Model saved to c:\Users\louis\Documents\Telerobo

INFO:tensorflow:Assets written to: C:\Users\louis\AppData\Local\Temp\tmpki4_x4x7\assets


Saved artifact at 'C:\Users\louis\AppData\Local\Temp\tmpki4_x4x7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 50, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  1629497834000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629497834768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629497835344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629497834192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629497835152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629497833808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629499540304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629499540496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629499539728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1629499539920: TensorSpec(shape=(), dtype=tf.resource, name=None)
